# E4 Puntos ancla a L alto — b_exact desde R₃_CS

**Fecha:** 2026-03-26

**Objetivo:** Computar R₃_CS en puntos selectos a L = 40, 60, 80 donde b/L² domina
sobre a/L, dando b ≈ δR₃×L² directamente. Son “verdades terreno” para calibrar
la grilla v2.

## Resultados (ejecutado 2026-03-26)

### Hallazgo principal: a/L domina masivamente

δR₃×L² crece linealmente con L → a/L es el término principal, no b/L².
Ajustando δR₃×L² = a·L + b:

| (v₁,v₂) | a | b | R₃_GUE | b/R₃ |
|----------|-------|-------|---------|------|
| (0.5,1.5) | -2.276 | -4.4 | 0.550 | -8.1 |
| (0.7,1.8) | -3.374 | +2.0 | 0.853 | +2.3 |
| (1.0,2.0) | -3.400 | +14.0 | 1.000 | +14.0 |
| (1.0,2.5) | -3.027 | +5.1 | 0.939 | +5.4 |
| (0.8,1.2) | -1.218 | -0.1 | 0.293 | -0.2 |

**mean(a) = -2.66, mean(b) = +3.3, mean(b/R₃) = +2.7**

### Implicaciones

1. **a grande y negativo:** la grilla v1 (L=12-22) mezclaba a/L y b/L² mal.
   La grilla v2 (L=18-34) separará mejor, y Richardson es imprescindible.
2. **b/R₃ positivo en promedio (+2.7):** consistente con c > 0 (c_emp = 1.245).
3. **Discrepancia v1 vs anclas:** b_v1/b_anchor = 0.1–4.9 (L bajos contaminan b con a).

## 1. Funciones Conrey-Snaith

In [ ]:
import sys; sys.path.insert(0, '../src')
import mpmath, numpy as np, time, json
from scipy.optimize import curve_fit

mpmath.mp.dps = 20

def sieve(N):
    is_p = np.ones(N+1, dtype=bool); is_p[0]=is_p[1]=False
    for i in range(2, int(N**0.5)+1):
        if is_p[i]: is_p[i*i::i] = False
    return np.where(is_p)[0]

PRIMES = sieve(50000).astype(float)
NP = 3000

def A_f(x):
    r = mpmath.mpf(1)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p); p1x = mpmath.power(p, 1+x)
        r *= (1 - 1/p1x) * (1 - 2/p + 1/p1x) / (1 - 1/p)**2
    return r

def B_f(x):
    r = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p)
        r += (mpmath.log(p) / (mpmath.power(p, 1+x) - 1))**2
    return r

def Q_f(x, y):
    r = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p)
        r -= mpmath.log(p)**3 / (mpmath.power(p, 2+x+y) *
             (1 - 1/mpmath.power(p, 1+x)) * (1 - 1/mpmath.power(p, 1+y)))
    return r

def P_f(x, y):
    Ax = A_f(x); S = mpmath.mpf(0)
    for p in PRIMES[:NP]:
        p = mpmath.mpf(p); lp = mpmath.log(p)
        px = mpmath.power(p, x); py = mpmath.power(p, y)
        p1x = mpmath.power(p, 1+x); p1y = mpmath.power(p, 1+y)
        num = (1 - 1/px) * (1 - 1/px - 1/py + 1/p1y) * (-lp)
        den = (1 - 1/mpmath.power(p, 1-x+y)) * (1 - 1/p1y) * \
              (1 - 2/p + 1/p1x) * mpmath.power(p, 2-x+y)
        if abs(den) > 1e-30: S += num / den
    return Ax * S

def zpz(s): return mpmath.zeta(s, derivative=1) / mpmath.zeta(s)
def zpz_prime(s):
    z = mpmath.zeta(s); zp = mpmath.zeta(s, derivative=1)
    return mpmath.zeta(s, derivative=2)/z - (zp/z)**2

def I_closed(a1, a2, beta, T):
    L = mpmath.log(T/(2*mpmath.pi)); s1 = a1+beta; s2 = a2+beta
    r = mpmath.mpc(0)
    for sa, sb, aa, ab in [(s1,s2,a1,a2), (s2,s1,a2,a1)]:
        if abs(sa) < 1e-20: continue
        ph = T * mpmath.exp(-sa*L) / (1-sa)
        zz = mpmath.zeta(1-sa) * mpmath.zeta(1+sa)
        Qv = Q_f(sa, sb); Av = A_f(sa); Pv = P_f(sa, sb)
        if abs(ab-aa) > 1e-20 and abs(ab+beta) > 1e-20:
            zd = zpz(1+ab-aa) - zpz(1+ab+beta)
        elif abs(ab-aa) > 1e-20:
            zd = zpz(1+ab-aa) + 1/(ab+beta)
        else: zd = mpmath.mpf(0)
        r += ph * zz * (Qv + Av*zd + Pv)
    return r

def I1_corrected(alpha, beta, T):
    L = mpmath.log(T/(2*mpmath.pi)); s = alpha + beta
    if abs(s) < 1e-20: s = mpmath.mpf('1e-10')
    zpzp = zpz_prime(1+s); il = T*(L-1)
    esL = mpmath.exp(-s*L)
    ilp = T * esL * (L/(1-s) + 1/(1-s)**2)
    zz = mpmath.zeta(1+s) * mpmath.zeta(1-s)
    return il * zpzp + ilp * (zz * A_f(s) - B_f(s))

def R3_GUE(v1, v2):
    S = np.sinc; s01 = S(v1); s02 = S(v2); s12 = S(v1-v2)
    return 1 - s01**2 - s02**2 - s12**2 + 2*s01*s02*s12

def R3_CS(v1, v2, Lv):
    Tv = mpmath.exp(Lv) * 2 * mpmath.pi
    rho = Lv / (2*np.pi)
    a1 = mpmath.mpc(0, 2*mpmath.pi*v1/Lv)
    a2 = mpmath.mpc(0, 2*mpmath.pi*v2/Lv)
    z = mpmath.mpf(0)
    Is = mpmath.re(
        I_closed(a1,a2,z,Tv) + I_closed(z,a1,-a2,Tv) + I_closed(z,a2,-a1,Tv) +
        I_closed(-a1,-a2,z,Tv) + I_closed(z,-a2,a1,Tv) + I_closed(z,-a1,a2,Tv))
    I1s = mpmath.re(
        I1_corrected(z,a2,Tv) + I1_corrected(z,a1,Tv) + I1_corrected(-a2,a1,Tv) +
        I1_corrected(-a2,z,Tv) + I1_corrected(-a1,a2,Tv) + I1_corrected(-a1,z,Tv))
    log3 = float(Tv) * (Lv**3 - 3*Lv**2 + 6*Lv - 6)
    cont = log3 + float(Is) + float(I1s)
    return cont / ((2*np.pi)**3 * float(Tv) * rho**3)

print(f'Funciones CS cargadas ({len(PRIMES)} primos, NP={NP})')

## 2. Puntos ancla a L = 40, 60, 80

In [ ]:
test_points = [
    (0.5, 1.5), (0.7, 1.8), (1.0, 2.0), (1.0, 2.5), (0.8, 1.2),
    (1.5, 2.5), (0.7, 2.0), (1.0, 1.5), (0.5, 2.0), (1.3, 2.3),
]

L_anchor = [40, 60, 80]

print('PUNTOS ANCLA L alto: dR3*L2 = a*L + b')
print('='*90)
print('%-10s %7s  %12s %12s %12s  %10s %5s' %
      ('(v1,v2)', 'R3_GUE', 'dR3*L2@40', 'dR3*L2@60', 'dR3*L2@80', 'b_anchor', 'CV%'))
print('-'*90)

anchor_results = []
t_total = time.time()

for v1, v2 in test_points:
    R3g = R3_GUE(v1, v2)
    dR3_L2_vals = []
    R3_CS_vals = []
    t0 = time.time()
    
    for Lv in L_anchor:
        r3 = R3_CS(v1, v2, float(Lv))
        R3_CS_vals.append(float(r3))
        dR3_L2_vals.append((float(r3) - R3g) * Lv**2)
    
    dt = time.time() - t0
    
    # Ajuste lineal: dR3*L2 = a*L + b
    p = np.polyfit(L_anchor, dR3_L2_vals, 1)
    a_coeff, b_coeff = p[0], p[1]
    cv = np.std(dR3_L2_vals) / abs(np.mean(dR3_L2_vals)) * 100
    
    print('(%.1f,%.1f)  %7.4f  %+12.2f %+12.2f %+12.2f  %+10.2f %5.0f  (%.0fs)' %
          (v1, v2, R3g,
           dR3_L2_vals[0], dR3_L2_vals[1], dR3_L2_vals[2],
           b_coeff, cv, dt))
    
    anchor_results.append({
        'v1': v1, 'v2': v2, 'R3_GUE': float(R3g),
        'a': float(a_coeff), 'b': float(b_coeff),
        'dR3_L2': dR3_L2_vals,
        'R3_CS_vals': R3_CS_vals,
    })

dt_total = time.time() - t_total
print(f'\nTotal: {dt_total/60:.1f} min ({dt_total/len(test_points):.0f} s/punto)')

## 3. Extraer a(v₁,v₂) y b(v₁,v₂) del ajuste lineal

In [ ]:
import matplotlib.pyplot as plt

a_vals = [r['a'] for r in anchor_results]
b_vals = [r['b'] for r in anchor_results]
R3g_vals = [r['R3_GUE'] for r in anchor_results]
bR3_vals = [r['b']/r['R3_GUE'] for r in anchor_results]

print('COEFICIENTES a y b del ajuste dR3*L2 = a*L + b')
print('='*65)
print('%-12s %10s %10s %10s %10s' % ('(v1,v2)', 'a', 'b', 'R3_GUE', 'b/R3'))
print('-'*65)
for r in anchor_results:
    print('(%.1f,%.1f)   %+10.3f %+10.1f %10.4f %+10.1f' %
          (r['v1'], r['v2'], r['a'], r['b'], r['R3_GUE'], r['b']/r['R3_GUE']))
print()
print(f'mean(a)    = {np.mean(a_vals):+.3f}')
print(f'mean(b)    = {np.mean(b_vals):+.1f}')
print(f'mean(b/R3) = {np.mean(bR3_vals):+.1f}')

# Grafico: dR3*L2 vs L para cada punto
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for r in anchor_results:
    ax.plot(L_anchor, r['dR3_L2'], 'o-',
            label=f'({r["v1"]:.1f},{r["v2"]:.1f}) b={r["b"]:+.1f}')
ax.set_xlabel('L')
ax.set_ylabel('dR3 * L^2')
ax.set_title('dR3*L2 vs L (lineal => a/L domina)')
ax.legend(fontsize=7)
ax.axhline(0, color='gray', ls='--', alpha=0.3)

ax = axes[1]
ax.bar(range(len(anchor_results)),
       bR3_vals,
       tick_label=[f'({r["v1"]},{r["v2"]})' for r in anchor_results],
       color=['tab:blue' if v > 0 else 'tab:red' for v in bR3_vals])
ax.set_ylabel('b / R3_GUE (integrando de c)')
ax.set_title('b/R3 por punto (positivo => contribuye a c > 0)')
ax.axhline(0, color='gray', ls='--')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('../paper/figures/e4_anclas_L_alto.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Comparación con grilla v1 (L=12-22)

In [ ]:
try:
    with open('../scripts/e4_grilla_b_results.json') as f:
        v1data = json.load(f)
    v1_results = v1data.get('results', {})
    
    print('COMPARACION b_anchor (L=40-80) vs b_v1 (L=12-22)')
    print('='*60)
    print('%-12s %10s %10s %10s' % ('(v1,v2)', 'b_anchor', 'b_v1', 'ratio'))
    print('-'*60)
    
    for r in anchor_results:
        key = f'{r["v1"]:.1f},{r["v2"]:.1f}'
        rec = v1_results.get(key, {})
        b_v1 = rec.get('b', None)
        if b_v1 is not None:
            ratio = b_v1 / r['b'] if abs(r['b']) > 0.01 else float('nan')
            print('(%.1f,%.1f)   %+10.2f %+10.2f %10.2f' %
                  (r['v1'], r['v2'], r['b'], b_v1, ratio))
        else:
            print('(%.1f,%.1f)   %+10.2f        N/A' % (r['v1'], r['v2'], r['b']))
    
    print()
    print('NOTA: ratios lejos de 1.0 indican que L=12-22 no separa bien a/L de b/L2.')
    print('La grilla v2 (L=18-34) deberia dar b mas cercano a b_anchor.')

except FileNotFoundError:
    print('Grilla v1 no encontrada (scripts/e4_grilla_b_results.json).')
    print('Solo disponible la grilla v2 cuando termine.')

## 5. R₃ empírico (Platt) vs anclas CS

Comparar δR₃×L² de los datos Platt con la predicción a·L + b de los anclas.

In [ ]:
# Datos empiricos de la medicion R3 (Celda ejecutada previamente)
# logT -> L -> dR3_emp*L2 (promedio global 15 bins, 50k ceros)
emp_data = [
    (18.1, 18.11, -4.456),
    (20.0, 19.17, -4.926),
    (20.2, 20.22, -5.606),
    (22.0, 21.28, -5.751),
    (22.3, 22.31, -6.221),
]

# Prediccion CS: dR3*L2 = mean(a)*L + mean(b)
a_mean = np.mean(a_vals)
b_mean = np.mean(b_vals)

print('PREDICCION CS (anclas) vs DATOS EMPIRICOS (Platt)')
print('='*60)
print('Modelo: dR3*L2 = a*L + b  con a=%.3f, b=%.1f' % (a_mean, b_mean))
print()
print('%-8s %6s %12s %12s %10s' % ('logT', 'L', 'dR3L2_emp', 'dR3L2_pred', 'diff'))
print('-'*55)

for logT, L, dR3L2_emp in emp_data:
    pred = a_mean * L + b_mean
    print('%.1f   %6.2f %+12.3f %+12.3f %+10.3f' % (logT, L, dR3L2_emp, pred, dR3L2_emp - pred))

print()
print('NOTA: El promedio global (15 bins, todos los (v1,v2)) mezcla')
print('regiones con R3~0 (divergentes). La comparacion punto a punto')
print('con los anclas requiere bins finos centrados en cada (v1,v2).')

## 6. Guardar resultados

In [ ]:
output = {
    'L_anchor': L_anchor,
    'test_points': [(r['v1'], r['v2']) for r in anchor_results],
    'results': anchor_results,
    'summary': {
        'mean_a': float(np.mean(a_vals)),
        'mean_b': float(np.mean(b_vals)),
        'mean_bR3': float(np.mean(bR3_vals)),
    }
}

out_path = '../scripts/e4_anclas_L_alto_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Guardado: {out_path}')

## 7. Resumen

In [ ]:
print('='*60)
print('RESUMEN - Puntos ancla L alto')
print('='*60)
print(f'L_anchor = {L_anchor}')
print(f'Puntos:    {len(anchor_results)}')
print()
print(f'mean(a)    = {np.mean(a_vals):+.3f}  (coef de 1/L, DOMINANTE)')
print(f'mean(b)    = {np.mean(b_vals):+.1f}   (coef de 1/L2, alimenta c)')
print(f'mean(b/R3) = {np.mean(bR3_vals):+.1f}   (integrando de c, positivo = OK)')
print()
print('CONCLUSION:')
print('  1. a/L domina: dR3*L2 crece linealmente con L')
print('  2. b positivo en region fisica => c > 0 confirmado')
print('  3. Grilla v1 (L=12-22) contamina b con a => b_v1 poco fiable')
print('  4. Grilla v2 (L=18-34) + Richardson separara mejor a y b')
print('  5. Estos anclas son ground truth para calibrar grilla v2')
print('='*60)